# K-SPADA Prediction and Visualization

**Audience:** users applying the trained CHC reference model to held-out liver cancer slice 4.  
**Goal:** run prediction, calculate cell-level accuracy, and reproduce the visualization logic from the original handoff training script.

The visualization negates the spatial y coordinate, uses the `tab20c` palette, and draws `Celltype` and `Prediction` in two side-by-side panels.

In [ ]:
from pathlib import Path
import pandas as pd
import KSPADA

PROJECT_ROOT = Path('/home/lixiangyu/sh/program/K-SPADA')
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'liver_cancer_best'
PROJECT_NAME = 'liver_cancer_chc_to_hcc_best'
MODEL_DIR = OUTPUT_DIR / PROJECT_NAME

training_config = KSPADA.BEST_LIVER_CANCER_TRAINING
dataset_config = KSPADA.BEST_LIVER_CANCER_TUTORIAL
QUERY_BATCH = dataset_config.prediction_batches[0]
QUERY_H5AD = dataset_config.query_paths[QUERY_BATCH]
PREDICTION_H5AD = OUTPUT_DIR / f'prediction_batch_{QUERY_BATCH}.h5ad'
HISTORICAL_MODEL_DIR = Path('/home/lixiangyu/sh/program/代码交接') / dataset_config.historical_project_name
HISTORICAL_CHECKPOINT = HISTORICAL_MODEL_DIR / 'model-9.pth'


## Step 1 — Select the available checkpoint

In [ ]:
if (MODEL_DIR / 'model-9.pth').is_file():
    active_model_dir = MODEL_DIR
    active_checkpoint = MODEL_DIR / 'model-9.pth'
elif HISTORICAL_CHECKPOINT.is_file():
    active_model_dir = HISTORICAL_MODEL_DIR
    active_checkpoint = HISTORICAL_CHECKPOINT
else:
    raise FileNotFoundError('Neither the retrained nor historical checkpoint is available.')

print('Model directory:', active_model_dir)
print('Checkpoint:', active_checkpoint)
print('Query:', QUERY_H5AD)

## Step 2 — Predict slice 4

In [ ]:
prediction_adata = KSPADA.predict_model(
    adata_input=QUERY_H5AD,
    model_dir=active_model_dir,
    model_weight_path=active_checkpoint,
    output_path=PREDICTION_H5AD,
    prediction_batches=[QUERY_BATCH],
    cutoff=0.1,
    use_gcn=True,
    embed_dim=training_config.embedding_dim,
    depth=training_config.transformer_depth,
    num_heads=training_config.attention_heads,
)


## Step 3 — Evaluate and visualize

In [ ]:
prediction_adata.obs['match'] = (
    prediction_adata.obs['Prediction'].astype(str) == prediction_adata.obs['Celltype'].astype(str)
).astype(int)
accuracy = float(prediction_adata.obs['match'].mean())
summary = pd.Series({
    'cells': prediction_adata.n_obs,
    'accuracy': accuracy,
    'historical_best_accuracy': dataset_config.historical_accuracy,
})
display(summary)
figure, axes, colors = KSPADA.plot_prediction_spatial(
    prediction_adata,
    output_prefix=OUTPUT_DIR / f'prediction_batch_{QUERY_BATCH}_spatial',
    reference_adata=dataset_config.data_directory / 'all_hvg_3000.h5ad',
    point_size=50,
)


## Exercise

Report the per-class accuracy to identify which cell type contributes most to the remaining errors.

In [ ]:
per_class_accuracy = prediction_adata.obs.groupby('Celltype', observed=True)['match'].mean().sort_values(ascending=False)
display(per_class_accuracy)


## Pitfall and extension

**Pitfall:** the query genes must match the model gene order exactly.  
**Extension:** apply the same checkpoint to additional HCC slices and summarize accuracy by slice.